In [1]:
# 1. 기본 설치 (최초 실행 시 한 번만)
!pip install -q streamlit
!pip install -q pyngrok
!pip install -q kobert-transformers
!pip install -q transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/

In [14]:
code = """
import streamlit as st
st.set_page_config(page_title="낚시성 기사 판별기", layout="centered")

import torch
import pandas as pd
from transformers import BertForSequenceClassification
from kobert_transformers import get_tokenizer

# 디바이스 설정 및 최대 길이
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 512

# 모델 및 토크나이저 로드
@st.cache_resource
def load_model():
    tokenizer = get_tokenizer()
    model = BertForSequenceClassification.from_pretrained('skt/kobert-base-v1', num_labels=2)
    model.load_state_dict(torch.load("kobert_clickbait_model.pt", map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return tokenizer, model

tokenizer, model = load_model()

# 예측 함수
def predict_clickbait(title, content):
    text = f"{title} [SEP] {content}"
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LEN,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(DEVICE)
    attention_mask = encoding['attention_mask'].to(DEVICE)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred = int(probs.argmax())  # 확률이 더 높은 클래스를 선택 (0: 낚시성, 1: 일반)
    return pred, probs

# Streamlit UI
st.title("🎣 낚시성 기사 판별기")
st.markdown("뉴스 기사의 제목과 본문을 입력하세요:")

title_input = st.text_input("제목을 입력하세요", "")
content_input = st.text_area("본문을 입력하세요", "", height=200)

if st.button("판별하기"):
    if title_input.strip() == "" or content_input.strip() == "":
        st.warning("제목과 본문을 모두 입력해주세요!")
    else:
        pred, probs = predict_clickbait(title_input, content_input)
        normal_prob = probs[0] * 100
        clickbait_prob = probs[1] * 100

        label = "✅ 일반 기사입니다" if pred == 0 else "🎣 낚시성 기사입니다"
        confidence = normal_prob if pred == 0 else clickbait_prob

        st.write(f"### {label}")
        st.write(f"**신뢰도: {confidence:.2f}%**")

        # 확률 바 차트 시각화
        prob_df = pd.DataFrame({
            "확률(%)": [normal_prob, clickbait_prob]
        }, index=["일반", "낚시성"])
        st.bar_chart(prob_df)
"""


In [15]:
# 2. 스트리밍 앱 실행을 위한 pyngrok 설정
from pyngrok import ngrok

# 3. Streamlit 앱 코드 저장
with open("app.py", "w") as f:
    f.write(code)

# 4. 학습된 모델 파일 업로드
from google.colab import files
print("📁 학습된 모델 파일(.pt)을 업로드하세요")
uploaded = files.upload()  # kobert_clickbait_model.pt 파일 업로드

📁 학습된 모델 파일(.pt)을 업로드하세요


Saving kobert_clickbait_model.pt to kobert_clickbait_model (2).pt


In [16]:
!ngrok config add-authtoken 2y5svBulvKLa2xKssrRN2iK5TcC_6XPDCeVSwj7r69d5bnmur

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [17]:
# 5. ngrok을 통해 웹 앱 실행
print("🚀 앱 실행 중... 몇 초만 기다리세요...")
public_url = ngrok.connect(8501)
print(f"✅ 접속 링크: {public_url}")

!streamlit run app.py &

🚀 앱 실행 중... 몇 초만 기다리세요...
✅ 접속 링크: NgrokTunnel: "https://5792-34-81-191-48.ngrok-free.app" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.81.191.48:8501

2025-06-12 12:57:22.250442: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749733042.346741   63348 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749733042.379906   63348 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-12 12:57:22.484571: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instru